<a href="https://colab.research.google.com/github/kolijayesh370-ops/walmart-supply-chain-engine/blob/main/Walmart_Predictive_Supply_Chain_Engine.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [60]:
# ==========================================
# STEP 1: INSTALL WEB INTERFACE CORE UTILITIES
# ==========================================
!pip install gradio scikit-learn pandas numpy -q

import os
import pandas as pd
import numpy as np
import gradio as gr
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error

print("🔄 Processing infrastructure configurations...")

# ==========================================
# STEP 2: DATA PROCESSING AND MODEL TRAINING
# ==========================================
if not os.path.exists("Walmart_Sales.csv"):
    raise FileNotFoundError("❌ Please upload 'Walmart_Sales.csv' to your left-hand sidebar files drawer before running.")

df = pd.read_csv("Walmart_Sales.csv")

# Standardize feature data structures and date formats
df['Date'] = pd.to_datetime(df['Date'], dayfirst=True)
if 'Holiday_Flag' in df.columns:
    df = df.rename(columns={'Holiday_Flag': 'IsHoliday'})
df['Store'] = df['Store'].astype(str)

# Deriving time features
df['Week_of_Year'] = df['Date'].dt.isocalendar().week.astype(int)
df['Year'] = df['Date'].dt.year

# Imputing missing values with column medians
for col in ['Weekly_Sales', 'Temperature', 'Fuel_Price', 'CPI', 'Unemployment']:
    if col in df.columns:
        df[col] = df[col].fillna(df[col].median())
df.dropna(inplace=True)
df.drop_duplicates(inplace=True)

# Trimming dataset outlier anomalies (99th percentile clamp)
q99 = df["Weekly_Sales"].quantile(0.99)
df = df[df["Weekly_Sales"] <= q99]

# Perform One-Hot categorical dummy encoding
df_features = df[["Store", "IsHoliday", "Temperature", "Fuel_Price", "CPI", "Unemployment", "Week_of_Year", "Year"]]
X = pd.get_dummies(df_features, columns=["Store"], drop_first=True)
y = df["Weekly_Sales"]

# Save explicit structural column mappings for inline testing
GLOBAL_MODEL_COLUMNS = X.columns.tolist()

# Fit Random Forest engine parameters
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
GLOBAL_RF_MODEL = RandomForestRegressor(n_estimators=30, random_state=42, n_jobs=-1)
GLOBAL_RF_MODEL.fit(X_train, y_train)

print(f"   📊 Model Training Finalized. R² Accuracy: {r2_score(y_test, GLOBAL_RF_MODEL.predict(X_test)):.4f}")

# ==========================================
# STEP 3: RUNTIME PREDICTION LOGIC ENGINE
# ==========================================
def run_live_prediction(store_id, is_holiday, temperature, fuel_price, cpi, unemployment, week, year):
    # Initialize zero-matrix aligned with original shape
    input_dataframe = pd.DataFrame(columns=GLOBAL_MODEL_COLUMNS)
    input_dataframe.loc[0] = 0.0

    # Process user form selections into features
    input_dataframe['IsHoliday'] = 1.0 if is_holiday == "Yes" else 0.0
    input_dataframe['Temperature'] = float(temperature)
    input_dataframe['Fuel_Price'] = float(fuel_price)
    input_dataframe['CPI'] = float(cpi)
    input_dataframe['Unemployment'] = float(unemployment)
    input_dataframe['Week_of_Year'] = int(week)
    input_dataframe['Year'] = int(year)

    # Extract structural categorical encoding for store IDs
    target_store_key = f"Store_{int(store_id)}"
    if target_store_key in input_dataframe.columns:
        input_dataframe[target_store_key] = 1.0

    # Execute random forest array optimization math
    calculated_sales = GLOBAL_RF_MODEL.predict(input_dataframe)[0]
    return f"${calculated_sales:,.2f}"

# ==========================================
# STEP 4: INTERACTIVE UI GENERATION
# ==========================================
print("🔄 Drawing final web interface...")

with gr.Blocks(theme=gr.themes.Soft()) as web_interface:
    gr.Markdown("# 📊 Walmart Predictive Supply Chain Engine")
    gr.Markdown("Adjust supply chain parameters below to run real-time automated stock optimization predictions inside your notebook.")

    with gr.Row():
        with gr.Column():
            gr.Markdown("### 🔮 Input Store Configurations")
            store_id = gr.Number(label="Select Store Location ID", value=1, minimum=1, maximum=45, precision=0)
            is_holiday = gr.Dropdown(label="Is this a holiday week?", choices=["No", "Yes"], value="No")
            temperature = gr.Slider(label="Predicted Temperature (°F)", minimum=0, maximum=100, value=55)
            fuel_price = gr.Number(label="Regional Fuel Price ($/Gallon)", value=3.25)
            cpi = gr.Number(label="Regional Consumer Price Index (CPI)", value=211.0)
            unemployment = gr.Slider(label="Regional Unemployment Rate (%)", minimum=2.0, maximum=15.0, value=7.5)
            week = gr.Slider(label="Calendar Week of the Year", minimum=1, maximum=52, value=25)
            year = gr.Dropdown(label="Target Year", choices=["2024", "2025", "2026"], value="2026")

            submit_btn = gr.Button("Generate Replenishment Prediction Orders", variant="primary")

        with gr.Column():
            gr.Markdown("### 🎯 Optimization Engine Output")
            output_display = gr.Textbox(label="Predicted Required Stock Demand Volume", placeholder="Awaiting parameter selection calculation...")

    # Wire component actions
    submit_btn.click(
        fn=run_live_prediction,
        inputs=[store_id, is_holiday, temperature, fuel_price, cpi, unemployment, week, year],
        outputs=output_display
    )

# Render everything directly inside your current notebook cell view
web_interface.launch(inline=True, share=False)

🔄 Processing infrastructure configurations...
   📊 Model Training Finalized. R² Accuracy: 0.9754
🔄 Drawing final web interface...


/tmp/ipykernel_10207/1839360263.py:91: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as web_interface:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
Note: opening Chrome Inspector may crash demo inside Colab notebooks.
* To create a public link, set `share=True` in `launch()`.


<IPython.core.display.Javascript object>